In [8]:
# Ensure Python Version 3.6 - 3.12!!!
!python --version

Python 3.11.13


In [9]:
!pip install -q syncode

In [10]:
from syncode import Syncode

# Define the grammar
We define the grammar for the problem we are solving. We consider the output to be constrained to two floats and an operator. We allow a new-line at the end to allow the model to "express" end of answer, the parser handles it via \s*.

In [11]:
grammar = r"""
    start: SP "(" NUM ")" OP "(" NUM ")" NL

    NUM: /-?(0|[1-9][0-9]{0,8})\.[0-9]{1,8}/
    OP: "+" | "-" | "*" | "/"
    SP: " "
    NL: "\n"
"""

# Load the Huggingface models
Syncode uses the Huggingface transformers library to load the models. Simply provide the HuggingFace ID of the model to be used.

In [12]:
import re, operator

model_name = "microsoft/phi-2"
syn_llm = Syncode(model=model_name, mode='grammar_mask', grammar=grammar,
                  parse_output_only=True, max_new_tokens=50)

[2026-10-28 02:46:38,445-root] - Loading model microsoft/phi-2 with device:cuda, device_map:auto, torch_dtype:torch.bfloat16
[2026-10-28 02:46:38,607-accelerate.utils.modeling] - We will use 90% of the memory on device 0 for storing the model, and 10% for the buffer to avoid OOM. You can set `max_memory` in to a higher value to use more memory (at your own risk).


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

[2026-10-28 02:46:47,269-syncode.mask_store.mask_store] - Using cache: True and fsm path cache/mask_stores/CodeGenTokenizerFast/grammar_mask_1764377865_50257.pkl exist: False
[2026-10-28 02:46:47,270-syncode.mask_store.mask_store] - Creating mask store for CodeGenTokenizerFast and custom, may take more than 10 minutes. Caching at /content/cache/mask_stores/CodeGenTokenizerFast/grammar_mask_1764377865_50257.pkl.
[2026-10-28 02:46:49,269-syncode.mask_store.fsm_set] - 6 FSMs with 34 states initialized in 0.02 seconds
[2026-10-28 02:46:49,272-syncode.mask_store.mask_store] - Ignore whitespace is False
[2026-10-28 02:46:49,273-syncode.mask_store.mask_store] - Number of 2 length terminal sequences reduced from 36 to 6


100%|██████████| 34/34 [00:15<00:00,  2.18it/s]

[2026-10-28 02:47:04,873-syncode.mask_store.mask_store] - Time taken to create mask store: 17.55 seconds


# Write the run_syncode function to do the operations.
We perform extraction using syncode and then use Python interpreter to solve the arithmetic. We provide few-shot examples.

In [13]:
FEWSHOT = (
    "Extract the two numbers and the operator as floats.\n"
    "Q: What is 5. plus 2.5?\nA: (5.0)+(2.5)\n"
    "Q: What is 10.0 divided by 4.?\nA: (10.0)/(4.0)\n"
    "Q: What is -4.5 multiplied by 2.0?\nA: (-4.5)*(2.0)\n"
    "Q: What is -7.0 minus -3.25?\nA: (-7.0)-(-3.25)\n"
)

OPS = {"+": operator.add, "-": operator.sub,
       "*": operator.mul, "/": operator.truediv}

PATTERN = re.compile(r"\s*(\(-?\d+\.\d+\))([+\-*/])(\(-?\d+\.\d+\))\s*")

def to_float(s):
    return float(s.strip("()"))

def run_syncode(textual_query):
    prompt = f"{FEWSHOT}Q: {textual_query}\nA:"
    raw = syn_llm.infer(prompt)[0]
    print("raw:", repr(raw))
    m = PATTERN.fullmatch(raw)
    if m is None:
        raise ValueError(f"Unexpected SynCode output: {raw!r}")
    a, op, b = to_float(m.group(1)), m.group(2), to_float(m.group(3))
    return ([a, b], op, float(OPS[op](a, b)))

In [14]:
TESTS = ["What is 327. multiplied by 11.0?",
          "What is 45.1 plus 23.54?",
          "What is 120.4 divided by 4.0?",
          "What is 90.5 minus 12.25?",
          "Multiply 3.2 by 4.0.",
          "What is -8.5 minus 2.0?",
          "What is -2.5 multiplied by -4.0?",
          "What is 6.0 minus -1.5?",
          "What is 99. divided by 4.?",
          "Subtract 3.5 from 10.0"]

for test in TESTS:
    print(run_syncode(test))

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


raw: ' (327.0)*(11.0)\n'
([327.0, 11.0], '*', 3597.0)
raw: ' (45.1)+(23.54)\n'
([45.1, 23.54], '+', 68.64)
raw: ' (120.4)/(4.0)\n'
([120.4, 4.0], '/', 30.1)
raw: ' (90.5)-(12.25)\n'
([90.5, 12.25], '-', 78.25)
raw: ' (3.2)*(4.0)\n'
([3.2, 4.0], '*', 12.8)
raw: ' (-8.5)-(2.0)\n'
([-8.5, 2.0], '-', -10.5)
raw: ' (-2.5)*(-4.0)\n'
([-2.5, -4.0], '*', 10.0)
raw: ' (6.0)-(-1.5)\n'
([6.0, -1.5], '-', 7.5)
raw: ' (99.0)/(4.0)\n'
([99.0, 4.0], '/', 24.75)
raw: ' (10.0)-(3.5)\n'
([10.0, 3.5], '-', 6.5)
